# Softmax: From Scores to Probabilities

A neural network outputs raw numbers ("logits"). How do we turn them into probabilities?

**Softmax** is the answer -- and it's the same idea whether we're classifying digits or predicting the next character in a sentence.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 12

## Part 1: The Problem

A neural network for MNIST outputs 10 raw scores (one per digit). These can be any real number -- positive, negative, large, small. How do we interpret them?

In [ ]:
# Raw scores (logits) from a neural network for digit classification
logits = torch.tensor([0.5, 0.2, 3.1, 1.0, -0.5, -1.0, 0.3, -0.2, 0.1, -0.8])
digits = list(range(10))

fig, ax = plt.subplots(figsize=(8, 3))
colors = ['#e74c3c' if i == 2 else '#3498db' for i in digits]
ax.barh(digits, logits.numpy(), color=colors)
ax.set_yticks(digits)
ax.set_yticklabels([str(d) for d in digits])
ax.set_xlabel('Raw score (logit)')
ax.set_title('Network output: raw scores for each digit')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f'Raw scores: {logits.tolist()}')
print(f'Highest score: digit {logits.argmax().item()} (score = {logits.max().item()})')
print()
print('But what does a score of 3.1 MEAN? Is the network 50% sure? 90%?')
print('We need probabilities!')

### Why not just normalize (divide by sum)?

Scores can be negative -- dividing by the sum doesn't work.

In [ ]:
# Naive approach: divide by sum
naive = logits / logits.sum()
print(f'Sum of logits: {logits.sum().item():.2f}')
print(f'Naive "probabilities": {naive.tolist()}')
print(f'Negative values! Not valid probabilities.')

## Part 2: Softmax

**Step 1:** Exponentiate (makes everything positive)

**Step 2:** Normalize (divide by sum so they add to 1)

$$P(\text{class } i) = \frac{e^{z_i}}{\sum_{j=1}^{k} e^{z_j}}$$

In [ ]:
# Step by step on a simple example: [2.0, 1.0, 0.1]
z = torch.tensor([2.0, 1.0, 0.1])

# Step 1: Exponentiate
exp_z = torch.exp(z)
print(f'Logits:          {z.tolist()}')
print(f'After exp:       {exp_z.tolist()}')
print(f'  (all positive!)')

# Step 2: Normalize
probs = exp_z / exp_z.sum()
print(f'After normalize: {[f"{p:.3f}" for p in probs.tolist()]}')
print(f'Sum = {probs.sum().item():.4f}  (sums to 1!)')

In [ ]:
# PyTorch has this built in
probs_pytorch = F.softmax(z, dim=0)
print(f'F.softmax: {[f"{p:.3f}" for p in probs_pytorch.tolist()]}')
print('Same result!')

In [ ]:
# Back to our MNIST example
probs = F.softmax(logits, dim=0)

print('MNIST digit classifier:')
print(f'{"Digit":<8} {"Logit":>8} {"exp(z)":>10} {"Probability":>12}')
print('=' * 40)
for i in range(10):
    marker = ' ←' if i == 2 else ''
    print(f'{i:<8} {logits[i].item():>8.1f} {torch.exp(logits[i]).item():>10.2f} {probs[i].item():>11.1%}{marker}')
print(f'{"":<8} {"":>8} {"":>10} {probs.sum().item():>11.1%}')
print(f'\nPrediction: digit {probs.argmax().item()} with {probs.max().item():.1%} confidence')

In [ ]:
# Visualize: logits vs probabilities
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#e74c3c' if i == 2 else '#3498db' for i in digits]

axes[0].barh(digits, logits.numpy(), color=colors)
axes[0].set_title('Raw logits', fontweight='bold')
axes[0].set_xlabel('Score')
axes[0].set_yticks(digits)
axes[0].invert_yaxis()

axes[1].barh(digits, probs.numpy(), color=colors)
axes[1].set_title('After softmax (probabilities)', fontweight='bold')
axes[1].set_xlabel('Probability')
axes[1].set_yticks(digits)
axes[1].invert_yaxis()
for i, p in enumerate(probs):
    if p > 0.05:
        axes[1].text(p.item() + 0.01, i, f'{p.item():.1%}', va='center', fontsize=10)

plt.suptitle('Softmax: from scores to probabilities', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

## Part 3: Softmax Amplifies Differences

Softmax doesn't just normalize -- it **amplifies** the gap between the largest score and the rest.

In [ ]:
# What happens as the scores get more confident?
# Multiply all logits by a scaling factor
scales = [0.5, 1.0, 2.0, 5.0]
base = torch.tensor([2.0, 1.0, 0.1])
labels = ['A', 'B', 'C']

fig, axes = plt.subplots(1, len(scales), figsize=(14, 3), sharey=True)

for ax, s in zip(axes, scales):
    p = F.softmax(base * s, dim=0)
    bars = ax.bar(labels, p.numpy(), color=['#e74c3c', '#3498db', '#2ecc71'])
    ax.set_title(f'scale = {s}', fontweight='bold')
    ax.set_ylim(0, 1)
    for bar, prob in zip(bars, p):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{prob.item():.1%}', ha='center', fontsize=10)

axes[0].set_ylabel('Probability')
plt.suptitle('Softmax amplifies as scores grow', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print('Small scale → nearly uniform (unsure)')
print('Large scale → nearly one-hot (very confident)')

## Part 4: Temperature

We can control this amplification with a **temperature** parameter $T$:

$$P(\text{class } i) = \frac{e^{z_i / T}}{\sum_{j} e^{z_j / T}}$$

- **Low T** (e.g., 0.1) → sharp, confident (picks the top choice)
- **T = 1** → standard softmax
- **High T** (e.g., 10) → flat, uncertain (more uniform)

In [ ]:
logits = torch.tensor([2.0, 1.0, 0.5, 0.1, -0.5])
labels = ['cat', 'dog', 'bird', 'fish', 'car']
temps = [0.1, 0.5, 1.0, 2.0, 10.0]

fig, axes = plt.subplots(1, len(temps), figsize=(16, 3.5), sharey=True)

for ax, T in zip(axes, temps):
    p = F.softmax(logits / T, dim=0)
    ax.bar(labels, p.numpy(), color='steelblue')
    ax.set_title(f'T = {T}', fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=45)

axes[0].set_ylabel('Probability')
plt.suptitle('Temperature controls sharpness of the distribution', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print('T → 0:  picks top choice with ~100% probability (greedy)')
print('T = 1:  standard softmax')
print('T → ∞:  uniform distribution (random)')

## Part 5: Cross-Entropy Loss

Once we have probabilities, we need a **loss function** to tell the network how wrong it is.

If the true class is $c$, and the network assigns probability $p_c$ to it:

$$\text{Loss} = -\log(p_c)$$

- If $p_c = 1.0$ (perfect) → loss = 0
- If $p_c = 0.01$ (very wrong) → loss = 4.6
- If $p_c \to 0$ → loss → $\infty$

In [ ]:
# Visualize: how loss changes with predicted probability
p = torch.linspace(0.01, 1.0, 100)
loss = -torch.log(p)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(p, loss, 'b-', linewidth=2)
ax.set_xlabel('Predicted probability for true class $p_c$')
ax.set_ylabel('Loss = $-\log(p_c)$')
ax.set_title('Cross-Entropy Loss', fontweight='bold')

# Annotate key points
ax.annotate('Perfect: loss = 0', xy=(1.0, 0), fontsize=11,
            xytext=(0.6, 1.0), arrowprops=dict(arrowstyle='->', color='green'),
            color='green', fontweight='bold')
ax.annotate('Very wrong: loss → ∞', xy=(0.05, 3.0), fontsize=11,
            xytext=(0.3, 3.5), arrowprops=dict(arrowstyle='->', color='red'),
            color='red', fontweight='bold')

ax.set_xlim(0, 1.05)
ax.set_ylim(-0.1, 5)
plt.tight_layout()
plt.show()

In [ ]:
# PyTorch: CrossEntropyLoss = softmax + (-log) in one step
logits = torch.tensor([[0.5, 0.2, 3.1, 1.0, -0.5, -1.0, 0.3, -0.2, 0.1, -0.8]])
true_label = torch.tensor([2])  # true digit is 2

# Manual: softmax then -log
probs = F.softmax(logits, dim=1)
loss_manual = -torch.log(probs[0, 2])

# PyTorch built-in (does both in one step, numerically stable)
loss_pytorch = F.cross_entropy(logits, true_label)

print(f'Probability of true class (digit 2): {probs[0, 2].item():.4f}')
print(f'Manual loss:  -log({probs[0, 2].item():.4f}) = {loss_manual.item():.4f}')
print(f'PyTorch loss: {loss_pytorch.item():.4f}')
print()
print('In practice, always use F.cross_entropy (includes softmax, more numerically stable).')

## Part 6: From Digits to Characters -- Preview of Next-Char Prediction

The same softmax idea applies to **predicting the next character** in a sentence!

Instead of 10 digit classes → we have ~27 character classes (a-z + space).

The network outputs a score for each character, softmax gives probabilities, and we sample the next character.

In [ ]:
# Imagine a network that predicts the next character after "the ca"
chars = list('abcdefghijklmnopqrstuvwxyz ')
n_chars = len(chars)

# Simulated logits (network output) -- 't' should be most likely
torch.manual_seed(42)
logits = torch.randn(n_chars) * 0.5
logits[chars.index('t')] = 3.0   # 't' (for "cat")
logits[chars.index('r')] = 1.5   # 'r' (for "car")
logits[chars.index('n')] = 1.0   # 'n' (for "can")
logits[chars.index('m')] = 0.8   # 'm' (for "cam")

probs = F.softmax(logits, dim=0)

# Show top predictions
top_k = 8
top_probs, top_idx = probs.topk(top_k)

print('Context: "the ca"  →  next character?')
print()
for i in range(top_k):
    c = chars[top_idx[i]]
    display_c = '⎵' if c == ' ' else c
    print(f'  "{display_c}" → P = {top_probs[i].item():.1%}   ("the ca{c}...")')
print(f'  ... (remaining {n_chars - top_k} chars share {(1 - top_probs.sum()).item():.1%})')

In [ ]:
# Visualize the probability distribution over all characters
fig, ax = plt.subplots(figsize=(12, 3.5))
display_chars = ['⎵' if c == ' ' else c for c in chars]
colors = ['#e74c3c' if c == 't' else '#e67e22' if c in 'rnm' else '#3498db' for c in chars]
ax.bar(display_chars, probs.numpy(), color=colors)
ax.set_ylabel('Probability')
ax.set_title('Next character prediction: "the ca___"', fontweight='bold')
plt.tight_layout()
plt.show()

### Temperature for text generation

In [ ]:
# Temperature controls creativity vs predictability
temps = [0.3, 1.0, 3.0]
temp_labels = ['T=0.3 (conservative)', 'T=1.0 (standard)', 'T=3.0 (creative)']

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)

for ax, T, label in zip(axes, temps, temp_labels):
    p = F.softmax(logits / T, dim=0)
    colors = ['#e74c3c' if c == 't' else '#e67e22' if c in 'rnm' else '#3498db' for c in chars]
    ax.bar(display_chars, p.numpy(), color=colors)
    ax.set_title(label, fontweight='bold')

axes[0].set_ylabel('Probability')
plt.suptitle('Temperature: "the ca___"', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print('Low temperature  → almost always picks "t" (safe, boring: "cat")')
print('High temperature → more variety (creative: could be "car", "can", "cam", ...)')

In [ ]:
# Sampling with different temperatures
torch.manual_seed(0)
n_samples = 20

for T in [0.3, 1.0, 3.0]:
    p = F.softmax(logits / T, dim=0)
    samples = torch.multinomial(p, n_samples, replacement=True)
    sampled_chars = [chars[i] for i in samples]
    print(f'T={T:<4} → {" ".join(sampled_chars)}')

## Summary

**Softmax** converts raw scores to a probability distribution:
$$P(i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

**Cross-entropy loss** measures how wrong the prediction is:
$$\text{Loss} = -\log(P(\text{true class}))$$

**Temperature** controls the sharpness of the distribution:
$$P(i) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

**The same framework works for:**
- Digit classification (10 classes)
- Character prediction (27 classes)
- Word/token prediction (50,000+ classes) ← this is how LLMs work!

**Next up:** Building a character-level language model that actually learns to predict the next character.